# Random forest tuning

One grid search per analysis, scored with grouped CV folds over the **training countries only** — the 20 countries `models.ipynb` holds out are never seen here. The winning hyperparameters are saved to `data/hyperparameters/rf_<name>.json`, which `models.ipynb` loads:

- `rf_union_density.json`
- `rf_electoral_democracy.json`
- `rf_electoral_democracy_no_vdem.json`

In [ ]:
import json
from datetime import date
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

In [ ]:
# NOTE: this block must stay identical to the one in models.ipynb, so the
# search never sees the test countries that models.ipynb holds out
df = pd.read_csv("../data/merged.csv")

ID_COLS = ["iso3", "country", "Unnamed: 0"]
DROP = ["emp_services_pct",     # emp shares sum to exactly 100
        "services_va_pct_gdp",  # value-added shares nearly sum to 100
        "emp_pop_ratio",        # ≈ lfp_rate × (1 − unemployment)
        "self_employed_pct",    # r = 0.996 with vulnerable_emp_pct
        "exports_pct_gdp",      # r = 0.987 with trade_pct_gdp
        "liberal_democracy"]    # r = 0.981 with electoral_democracy
# V-Dem builds electoral_democracy (v2x_polyarchy) from freedom_of_association
# (v2x_frassoc_thick), which is itself built from cso_entry_exit and
# cso_repression; civil_society_index shares those two inputs (codebook v16).
# Predicting democracy from them would be predicting the index from its parts
DEM_INGREDIENTS = ["freedom_of_association", "cso_entry_exit",
                   "cso_repression", "civil_society_index"]
# every V-Dem column in merged.csv (VDEM_NAMES in cleaning.ipynb), for the
# democracy model with no V-Dem predictors at all
VDEM_COLS = ["freedom_of_association", "civil_society_index", "electoral_democracy",
             "liberal_democracy", "cso_repression", "cso_entry_exit",
             "egalitarian_component", "equal_resource_distribution",
             "power_by_socioeconomic_position"]

cols_union = [c for c in df if c not in ID_COLS + ["union_density"] + DROP]
cols_dem = [c for c in df if c not in ID_COLS + ["electoral_democracy"] + DROP + DEM_INGREDIENTS]
cols_dem_novdem = [c for c in df if c not in ID_COLS + VDEM_COLS + DROP]

df = df.dropna().reset_index(drop=True)

random_state = 42

# one train/test split: models.ipynb fits its OLS summaries on the training
# rows, and tuning.ipynb tunes on the training countries only. whole countries
# are held out, so no country appears in both halves
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
train_idx, test_idx = next(splitter.split(df, groups=df["iso3"]))
train, test = df.loc[train_idx], df.loc[test_idx]

PARAMS_DIR = Path("../data/hyperparameters")

PARAMS_DIR.mkdir(parents=True, exist_ok=True)

print(f"tuning on {len(train)} rows / {train['iso3'].nunique()} countries")
print(f"features -- union density: {len(cols_union)} | democracy: {len(cols_dem)} | "
      f"democracy, no V-Dem: {len(cols_dem_novdem)}")

In [ ]:
# folds group by country as well: a country is never in a fold's train and
# validation halves at once, which is how models.ipynb will be judged
GRID = {
    "max_features": [0.2, 0.33, 0.5, 1.0],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_samples": [None, 0.7],
}
N_ESTIMATORS = 1000


def tune(target, feature_cols, name=None):
    # `name` keys the saved file, so two analyses of one target don't collide
    name = name or target
    X = train[feature_cols]
    y = train[target]
    groups = train["iso3"]

    scaler = StandardScaler().fit(X)
    X_s = pd.DataFrame(scaler.transform(X), columns=feature_cols, index=X.index)

    search = GridSearchCV(
        RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=random_state, n_jobs=-1),
        GRID,
        cv=GroupKFold(5),
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
    ).fit(X_s, y, groups=groups)

    results = pd.DataFrame(search.cv_results_)
    results["rmse"] = -results["mean_test_score"]
    best = results.loc[results["rmse"].idxmin()]

    print(f"{name}: {len(results)} combinations tried\n")
    print(results.nsmallest(10, "rmse")[
        ["param_max_features", "param_min_samples_leaf", "param_max_samples", "rmse", "std_test_score"]
    ].to_string(index=False))

    # if the spread across all combinations is small next to the fold-to-fold
    # noise, trust the direction the winner points, not its exact values
    print(f"\nRMSE across all combinations: {results.rmse.min():.3f} to {results.rmse.max():.3f}"
          f"  (a spread of {results.rmse.max() - results.rmse.min():.3f})")
    print(f"fold-to-fold spread within the best combination: {best['std_test_score']:.3f}")

    export = {
        "name": name,
        "target": target,
        "features": list(feature_cols),
        "params": {"n_estimators": N_ESTIMATORS, **search.best_params_},
        "cv_rmse": round(float(-search.best_score_), 4),
        "cv_rmse_sd": round(float(best["std_test_score"]), 4),
        "searched": {k: [str(v) for v in vals] for k, vals in GRID.items()},
        "n_combinations": int(len(results)),
        "train_rows": int(len(train)),
        "train_countries": int(groups.nunique()),
        "random_state": random_state,
        "created": date.today().isoformat(),
    }

    path = PARAMS_DIR / f"rf_{name}.json"
    with open(path, "w") as f:
        json.dump(export, f, indent=2)

    print(f"\nsaved to {path}: {export['params']}")
    return results

## Union density

In [ ]:
results_union = tune("union_density", cols_union)

## Electoral democracy

The four `DEM_INGREDIENTS` features are excluded, and `union_density` is one of the predictors.

In [ ]:
results_dem = tune("electoral_democracy", cols_dem)

## Electoral democracy, no V-Dem predictors

Same target, but every V-Dem column (`VDEM_COLS`) is removed from the predictors — World Bank variables and `union_density` only. Saved under its own name so it doesn't overwrite the section above.

In [ ]:
results_novdem = tune("electoral_democracy", cols_dem_novdem, name="electoral_democracy_no_vdem")